<a href="https://colab.research.google.com/github/yosungcho/yosungcho.github.io/blob/main/090826AMDTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
============================================================================
AMD Binary Classification Head Training on ODIR-5K — Google Colab Script
============================================================================

Purpose:
    Train an EfficientNet-B2 binary classifier to detect Age-related
    Macular Degeneration (AMD) from fundus photographs (ODIR-5K dataset).
    This becomes the "AMD head" of the FundusView multi-head oculomic
    platform, sitting alongside existing DR, glaucoma, cataract, and
    (future) retinal age heads.

Assumed setup:
    - This script lives at:
        /content/drive/MyDrive/FundusView/amd/amd_training_colab.py
    - ODIR-5K data lives at:
        /content/drive/MyDrive/FundusView/ODIR-5K/
      Structure depends on which Kaggle version was downloaded. This
      script handles the most common layout:
        - preprocessed_images/  OR  Training_Images/  (JPEG files)
        - full_df.csv           OR  data.xlsx         (labels)
      Column names and paths auto-detected in Section 3.
    - Colab notebook mounts Drive, then runs this script.

Design notes:
    - Binary output (AMD absent / AMD present) matches the existing
      parallel binary CNN pattern in FundusView (DR, glaucoma, cataract).
    - Three-category uncertainty routing applied at inference:
        p < low_threshold  -> Normal
        low <= p < high    -> Uncertain
        p >= high          -> AMD present (referral recommended)
    - Handles ODIR-5K's known ~5-7% AMD prevalence via class-weighted loss.
    - Uses patient-level split (not image-level) to prevent leakage from
      paired left/right eye images.

Author: Yosung
Project: FundusView / Jojo CNN oculomic platform extension
============================================================================
"""

# ============================================================================
# SECTION 1: SETUP AND CONFIGURATION
# ============================================================================

import os
import sys
import glob
import json
import time
from datetime import datetime

# ---------- Path configuration (edit if your Drive structure differs) ----------
DRIVE_ROOT = '/content/drive/MyDrive/FundusView'
ODIR_DIR = os.path.join(DRIVE_ROOT, 'ODIR-5K')

# Image directory and labels file will be auto-detected in Section 3 since
# Kaggle uploads of ODIR-5K vary. Common candidates:
IMAGE_DIR_CANDIDATES = [
    os.path.join(ODIR_DIR, 'preprocessed_images'),
    os.path.join(ODIR_DIR, 'Training_Images'),
    os.path.join(ODIR_DIR, 'ODIR-5K', 'Training Images'),
    os.path.join(ODIR_DIR, 'images'),
]
LABEL_FILE_CANDIDATES = [
    os.path.join(ODIR_DIR, 'full_df.csv'),
    os.path.join(ODIR_DIR, 'data.xlsx'),
    os.path.join(ODIR_DIR, 'ODIR-5K_Training_Annotations.xlsx'),
    os.path.join(ODIR_DIR, 'labels.csv'),
]

PROJECT_DIR = os.path.join(DRIVE_ROOT, 'amd')
CHECKPOINT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_DIR, 'results')
LOG_DIR = os.path.join(PROJECT_DIR, 'logs')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(LOG_DIR, exist_ok=True)

# ---------- Training configuration ----------
CONFIG = {
    'backbone': 'efficientnet_b2',         # matches Cheung 2022 and your retinal age head
    'image_size': 512,
    'batch_size': 16,                      # good for T4 (16GB); drop to 16 if OOM
    'num_epochs': 40,                      # more than age model — smaller dataset
    'learning_rate': 1e-4,
    'weight_decay': 1e-4,
    'num_workers': 2,
    'seed': 42,
    'val_split': 0.15,
    'test_split': 0.15,
    'use_pretrained': True,
    'dropout': 0.3,
    'early_stopping_patience': 10,         # patience matters more for imbalanced data
    'uncertain_lower_threshold': 0.30,     # p < 0.30 -> Normal
    'uncertain_upper_threshold': 0.60,     # p >= 0.60 -> AMD present
                                           # in between -> Uncertain
    'oversample_amd': True,                # sampler upweights AMD-positive patients
}

# ---------- Timestamp for this run ----------
RUN_ID = datetime.now().strftime('%Y%m%d_%H%M%S')
RUN_LOG = os.path.join(LOG_DIR, f'run_{RUN_ID}.log')

def log(msg):
    """Print to stdout and append to log file on Drive (survives disconnects)."""
    timestamp = datetime.now().strftime('%H:%M:%S')
    line = f"[{timestamp}] {msg}"
    print(line)
    with open(RUN_LOG, 'a') as f:
        f.write(line + '\n')

log(f"=== AMD training run {RUN_ID} started ===")
log(f"Config: {json.dumps(CONFIG, indent=2)}")


# ============================================================================
# SECTION 2: DEPENDENCIES
# ============================================================================

os.system('pip install timm grad-cam openpyxl --quiet')

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve
)
from tqdm import tqdm
import matplotlib.pyplot as plt
import timm

# Reproducibility
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    log(f"GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    log("WARNING: No GPU detected. Enable GPU: Runtime > Change runtime type > T4 GPU")


# ============================================================================
# SECTION 3: DATA PREPARATION
# ============================================================================

# ---------- Auto-detect image directory ----------
IMAGE_DIR = None
for candidate in IMAGE_DIR_CANDIDATES:
    if os.path.isdir(candidate):
        IMAGE_DIR = candidate
        break
if IMAGE_DIR is None:
    log(f"ERROR: No image directory found. Checked: {IMAGE_DIR_CANDIDATES}")
    log(f"Please add your ODIR image folder path to IMAGE_DIR_CANDIDATES in Section 1")
    raise FileNotFoundError("ODIR image directory not found")
log(f"Using image directory: {IMAGE_DIR}")

# ---------- Auto-detect and load labels ----------
LABEL_FILE = None
for candidate in LABEL_FILE_CANDIDATES:
    if os.path.isfile(candidate):
        LABEL_FILE = candidate
        break
if LABEL_FILE is None:
    log(f"ERROR: No label file found. Checked: {LABEL_FILE_CANDIDATES}")
    raise FileNotFoundError("ODIR label file not found")
log(f"Using label file: {LABEL_FILE}")

if LABEL_FILE.endswith('.xlsx'):
    df = pd.read_excel(LABEL_FILE)
else:
    df = pd.read_csv(LABEL_FILE)

log(f"Loaded labels: {len(df)} rows")
log(f"Columns: {list(df.columns)}")

# ---------- Column detection ----------
# ODIR-5K has variations across Kaggle uploads. Handle the two most common:
#
# Version 1 (andrewmvd upload — most common): image-level rows with columns:
#   filename, labels (list of classes like "['N']"), and separate one-hot columns
#   like N, D, G, C, A, H, M, O
#
# Version 2 (original challenge format): patient-level rows with columns:
#   ID, Patient Age, Patient Sex, Left-Fundus, Right-Fundus,
#   Left-Diagnostic Keywords, Right-Diagnostic Keywords, N, D, G, C, A, H, M, O

AMD_COLUMN = 'A'  # AMD one-hot column, standard in both formats

# Determine data layout
if 'Left-Fundus' in df.columns and 'Right-Fundus' in df.columns:
    log("Detected patient-level format (original challenge structure)")
    # Reshape from patient-level to image-level by melting left/right
    left_df = df[['ID', 'Left-Fundus', AMD_COLUMN]].copy()
    left_df.columns = ['patient_id', 'filename', 'amd']
    left_df['eye'] = 'left'

    right_df = df[['ID', 'Right-Fundus', AMD_COLUMN]].copy()
    right_df.columns = ['patient_id', 'filename', 'amd']
    right_df['eye'] = 'right'

    df = pd.concat([left_df, right_df], ignore_index=True)
elif 'filename' in df.columns:
    log("Detected image-level format")
    # Need to derive patient_id from filename (e.g., "1234_left.jpg" -> patient 1234)
    if 'ID' in df.columns:
        df = df.rename(columns={'ID': 'patient_id'})
    else:
        df['patient_id'] = df['filename'].str.extract(r'(\d+)').astype(int)
    df = df[['patient_id', 'filename', AMD_COLUMN]].copy()
    df.columns = ['patient_id', 'filename', 'amd']
else:
    log(f"ERROR: Could not detect ODIR format from columns: {list(df.columns)}")
    log("Please edit Section 3 to match your label file structure")
    raise ValueError("Unknown ODIR label format")

# Convert AMD column to int (may be int, str, or bool depending on source)
df['amd'] = df['amd'].astype(int)

# Drop rows with missing filenames
before = len(df)
df = df.dropna(subset=['filename']).copy()
df = df[df['filename'].astype(str).str.strip() != ''].copy()
log(f"Dropped {before - len(df)} rows with missing filenames; {len(df)} images remain")
# Filter out rows whose image files don't actually exist on disk
# (Larxel's ODIR-5K version is missing ~2.5% of files referenced in CSV)
log("Verifying image files exist on disk...")
existing_files = set(os.listdir(IMAGE_DIR))
before = len(df)
df = df[df['filename'].astype(str).isin(existing_files)].reset_index(drop=True)
dropped = before - len(df)
log(f"Filtered {dropped} rows whose image files don't exist ({dropped/before*100:.1f}%); {len(df)} images remain")

# ---------- Class distribution ----------
amd_count = df['amd'].sum()
normal_count = len(df) - amd_count
amd_pct = amd_count / len(df) * 100
log(f"Class distribution: {amd_count} AMD ({amd_pct:.1f}%), {normal_count} non-AMD ({100-amd_pct:.1f}%)")
log(f"Imbalance ratio: {normal_count/amd_count:.1f}:1")

# ---------- Verify a few images exist ----------
sample_check = df.sample(min(10, len(df)), random_state=42)
missing = 0
for _, row in sample_check.iterrows():
    img_path = os.path.join(IMAGE_DIR, str(row['filename']))
    if not os.path.exists(img_path):
        if not any(os.path.exists(img_path.replace('.jpg', ext)) for ext in ['.jpeg', '.png', '.JPG']):
            missing += 1
if missing > 0:
    log(f"WARNING: {missing}/10 sampled image files not found. Check IMAGE_DIR.")
    log(f"Example expected path: {os.path.join(IMAGE_DIR, sample_check.iloc[0]['filename'])}")

# ---------- Patient-level split (critical for paired eye images) ----------
log("Splitting by patient_id to prevent left/right eye leakage across splits...")

gss_test = GroupShuffleSplit(n_splits=1, test_size=CONFIG['test_split'], random_state=CONFIG['seed'])
train_val_idx, test_idx = next(gss_test.split(df, groups=df['patient_id']))
df_trainval = df.iloc[train_val_idx].reset_index(drop=True)
df_test = df.iloc[test_idx].reset_index(drop=True)

val_frac_of_trainval = CONFIG['val_split'] / (1 - CONFIG['test_split'])
gss_val = GroupShuffleSplit(n_splits=1, test_size=val_frac_of_trainval, random_state=CONFIG['seed'])
train_idx, val_idx = next(gss_val.split(df_trainval, groups=df_trainval['patient_id']))
df_train = df_trainval.iloc[train_idx].reset_index(drop=True)
df_val = df_trainval.iloc[val_idx].reset_index(drop=True)

log(f"Train: {len(df_train)} images, {df_train['patient_id'].nunique()} patients, {df_train['amd'].sum()} AMD")
log(f"Val:   {len(df_val)} images, {df_val['patient_id'].nunique()} patients, {df_val['amd'].sum()} AMD")
log(f"Test:  {len(df_test)} images, {df_test['patient_id'].nunique()} patients, {df_test['amd'].sum()} AMD")

# Verify no patient overlap
tr, va, te = set(df_train['patient_id']), set(df_val['patient_id']), set(df_test['patient_id'])
assert len(tr & va) == 0 and len(tr & te) == 0 and len(va & te) == 0, "PATIENT LEAKAGE"
log("Patient split verified: no leakage")

# Save splits
df_train.to_csv(os.path.join(RESULTS_DIR, f'split_train_{RUN_ID}.csv'), index=False)
df_val.to_csv(os.path.join(RESULTS_DIR, f'split_val_{RUN_ID}.csv'), index=False)
df_test.to_csv(os.path.join(RESULTS_DIR, f'split_test_{RUN_ID}.csv'), index=False)


# ============================================================================
# SECTION 4: DATASET AND TRANSFORMS
# ============================================================================

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# Stronger augmentation than age model — AMD needs invariance to
# imaging variability more than age does
train_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),          # macular pathology is orientation-invariant
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['image_size'], CONFIG['image_size'])),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class ODIRAMDDataset(Dataset):
    def __init__(self, df, image_dir, transform=None):
        self.df = df.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        filename = str(row['filename'])

        # Try direct path first, then common extension variations
        image_path = os.path.join(self.image_dir, filename)
        if not os.path.exists(image_path):
            base, _ = os.path.splitext(image_path)
            for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
                if os.path.exists(base + ext):
                    image_path = base + ext
                    break
            else:
                raise FileNotFoundError(f"Image not found: {filename}")

        image = Image.open(image_path).convert('RGB')
        if self.transform:
            image = self.transform(image)

        label = torch.tensor(row['amd'], dtype=torch.float32)
        return image, label


train_dataset = ODIRAMDDataset(df_train, IMAGE_DIR, transform=train_transform)
val_dataset = ODIRAMDDataset(df_val, IMAGE_DIR, transform=eval_transform)
test_dataset = ODIRAMDDataset(df_test, IMAGE_DIR, transform=eval_transform)

log("Loading one sample to verify dataset works...")
sample_img, sample_label = train_dataset[0]
log(f"Sample image tensor shape: {sample_img.shape}, label: {sample_label.item()}")

# ---------- Weighted sampler to handle class imbalance ----------
if CONFIG['oversample_amd']:
    # Each AMD-positive sample gets weight proportional to (1/prevalence)
    # so the batch composition becomes closer to 50/50
    train_labels = df_train['amd'].values
    class_counts = np.array([len(train_labels) - train_labels.sum(), train_labels.sum()])
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(
        weights=sample_weights,
        num_samples=len(sample_weights),
        replacement=True
    )
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              sampler=sampler, num_workers=CONFIG['num_workers'],
                              pin_memory=True)
    log(f"Using WeightedRandomSampler: AMD samples oversampled to balance batches")
else:
    train_loader = DataLoader(train_dataset, batch_size=CONFIG['batch_size'],
                              shuffle=True, num_workers=CONFIG['num_workers'],
                              pin_memory=True)

val_loader = DataLoader(val_dataset, batch_size=CONFIG['batch_size'],
                        shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=CONFIG['batch_size'],
                         shuffle=False, num_workers=CONFIG['num_workers'], pin_memory=True)


# ============================================================================
# SECTION 5: MODEL
# ============================================================================

class AMDBinaryModel(nn.Module):
    """
    EfficientNet-B2 backbone with binary classification head for AMD.
    Output is a single logit (apply sigmoid at inference for probability).
    Designed to slot into FundusView multi-head architecture as one of
    the parallel binary CNN heads.
    """
    def __init__(self, backbone_name='efficientnet_b2', pretrained=True, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )
        backbone_dim = self.backbone.num_features

        self.amd_head = nn.Sequential(
            nn.Linear(backbone_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 1)
        )

    def forward(self, x):
        features = self.backbone(x)
        logit = self.amd_head(features)
        return logit.squeeze(-1)


model = AMDBinaryModel(
    backbone_name=CONFIG['backbone'],
    pretrained=CONFIG['use_pretrained'],
    dropout=CONFIG['dropout']
).to(device)

n_params = sum(p.numel() for p in model.parameters())
log(f"Model: {CONFIG['backbone']}, {n_params:,} parameters")


# ============================================================================
# SECTION 6: TRAINING SETUP
# ============================================================================

optimizer = AdamW(model.parameters(),
                  lr=CONFIG['learning_rate'],
                  weight_decay=CONFIG['weight_decay'])
scheduler = CosineAnnealingLR(optimizer, T_max=CONFIG['num_epochs'])

# BCE with logits — numerically stable, handles binary classification directly
# pos_weight upweights the positive class (AMD) in the loss when not using oversampling
if CONFIG['oversample_amd']:
    criterion = nn.BCEWithLogitsLoss()
else:
    pos_weight = torch.tensor([normal_count / max(amd_count, 1)]).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    log(f"Using pos_weight={pos_weight.item():.2f} to handle class imbalance")


# ============================================================================
# SECTION 7: RESUME FROM CHECKPOINT
# ============================================================================

def find_latest_checkpoint():
    pattern = os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_*.pth')
    checkpoints = sorted(glob.glob(pattern))
    return checkpoints[-1] if checkpoints else None


start_epoch = 0
best_val_auroc = 0.0
epochs_without_improvement = 0
history = {'train_loss': [], 'val_loss': [], 'val_auroc': [], 'val_auprc': [], 'lr': []}

latest_ckpt = find_latest_checkpoint()
if latest_ckpt:
    log(f"Found existing checkpoint: {latest_ckpt}")
    log("Resuming training. To start fresh, delete files in CHECKPOINT_DIR.")
    ckpt = torch.load(latest_ckpt, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    best_val_auroc = ckpt['best_val_auroc']
    epochs_without_improvement = ckpt.get('epochs_without_improvement', 0)
    history = ckpt.get('history', history)
    log(f"Resuming from epoch {start_epoch}, best val AUROC so far: {best_val_auroc:.4f}")
else:
    log("No checkpoint found; starting fresh training")


# ============================================================================
# SECTION 8: TRAINING LOOP
# ============================================================================

def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    losses = []
    pbar = tqdm(loader, desc="Train", leave=False)
    for images, labels in pbar:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.3f}'})
    return sum(losses) / len(losses)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    losses = []
    all_logits, all_labels = [], []
    for images, labels in tqdm(loader, desc="Eval", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        losses.append(loss.item())
        all_logits.extend(logits.cpu().numpy().tolist())
        all_labels.extend(labels.cpu().numpy().tolist())

    all_logits = np.array(all_logits)
    all_labels = np.array(all_labels)
    all_probs = 1 / (1 + np.exp(-all_logits))  # sigmoid

    avg_loss = sum(losses) / len(losses)

    # Only compute AUROC/AUPRC if both classes present in eval set
    if len(np.unique(all_labels)) > 1:
        auroc = roc_auc_score(all_labels, all_probs)
        auprc = average_precision_score(all_labels, all_probs)
    else:
        auroc = auprc = float('nan')

    return avg_loss, auroc, auprc, all_probs, all_labels


log(f"=== Starting training from epoch {start_epoch} to {CONFIG['num_epochs']} ===")
training_start_time = time.time()

for epoch in range(start_epoch, CONFIG['num_epochs']):
    epoch_start = time.time()

    train_loss = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_auroc, val_auprc, _, _ = evaluate(model, val_loader, criterion, device)

    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_auroc'].append(val_auroc)
    history['val_auprc'].append(val_auprc)
    history['lr'].append(current_lr)

    epoch_time = time.time() - epoch_start
    log(f"Epoch {epoch+1}/{CONFIG['num_epochs']} | "
        f"Train loss: {train_loss:.4f} | Val loss: {val_loss:.4f} | "
        f"Val AUROC: {val_auroc:.4f} | Val AUPRC: {val_auprc:.4f} | "
        f"LR: {current_lr:.2e} | Time: {epoch_time:.0f}s")

    # Save epoch checkpoint (for resume)
    ckpt_path = os.path.join(CHECKPOINT_DIR, f'checkpoint_epoch_{epoch:03d}.pth')
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'best_val_auroc': best_val_auroc,
        'epochs_without_improvement': epochs_without_improvement,
        'history': history,
        'config': CONFIG,
    }, ckpt_path)

    # Save best model (selected by AUROC — best metric for imbalanced binary)
    if val_auroc > best_val_auroc:
        best_val_auroc = val_auroc
        epochs_without_improvement = 0
        best_path = os.path.join(CHECKPOINT_DIR, 'best_model.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_auroc': val_auroc,
            'val_auprc': val_auprc,
            'config': CONFIG,
        }, best_path)
        log(f"  → New best val AUROC {val_auroc:.4f} saved to best_model.pth")
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= CONFIG['early_stopping_patience']:
            log(f"Early stopping: no AUROC improvement for {CONFIG['early_stopping_patience']} epochs")
            break

    # Housekeeping
    all_epoch_ckpts = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'checkpoint_epoch_*.pth')))
    for old_ckpt in all_epoch_ckpts[:-3]:
        os.remove(old_ckpt)

total_time = time.time() - training_start_time
log(f"=== Training complete in {total_time/60:.1f} minutes ===")
log(f"Best validation AUROC: {best_val_auroc:.4f}")


# ============================================================================
# SECTION 9: TEST SET EVALUATION
# ============================================================================

log("=== Loading best model for test evaluation ===")
best_ckpt = torch.load(os.path.join(CHECKPOINT_DIR, 'best_model.pth'), map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
log(f"Loaded best model from epoch {best_ckpt['epoch']} with val AUROC {best_ckpt['val_auroc']:.4f}")

test_loss, test_auroc, test_auprc, test_probs, test_labels = evaluate(
    model, test_loader, criterion, device
)
log(f"Test AUROC: {test_auroc:.4f}")
log(f"Test AUPRC: {test_auprc:.4f}")

# Three-category warning tier assignment (matching FundusView convention)
low_t = CONFIG['uncertain_lower_threshold']
high_t = CONFIG['uncertain_upper_threshold']
tiers = np.where(test_probs < low_t, 'Normal',
        np.where(test_probs >= high_t, 'AMD', 'Uncertain'))
tier_counts = pd.Series(tiers).value_counts()
log(f"=== Test set tier distribution ===")
for tier in ['Normal', 'Uncertain', 'AMD']:
    count = tier_counts.get(tier, 0)
    pct = count / len(tiers) * 100
    log(f"  {tier}: {count} ({pct:.1f}%)")

# Confusion matrix at binary threshold of 0.5 (standard operating point)
predictions_binary = (test_probs >= 0.5).astype(int)
cm = confusion_matrix(test_labels.astype(int), predictions_binary)
tn, fp, fn, tp = cm.ravel()
sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
ppv = tp / (tp + fp) if (tp + fp) > 0 else 0
npv = tn / (tn + fn) if (tn + fn) > 0 else 0

log(f"=== Test performance at threshold=0.5 ===")
log(f"  Confusion matrix:")
log(f"    TN={tn}  FP={fp}")
log(f"    FN={fn}  TP={tp}")
log(f"  Sensitivity (Recall): {sensitivity:.3f}")
log(f"  Specificity:          {specificity:.3f}")
log(f"  PPV (Precision):      {ppv:.3f}")
log(f"  NPV:                  {npv:.3f}")

# Find operating point at 95% specificity (screening-appropriate)
fpr, tpr, thresholds_roc = roc_curve(test_labels, test_probs)
idx_95spec = np.argmin(np.abs((1 - fpr) - 0.95))
sens_at_95spec = tpr[idx_95spec]
thr_at_95spec = thresholds_roc[idx_95spec]
log(f"  Sensitivity at 95% specificity: {sens_at_95spec:.3f} (threshold={thr_at_95spec:.3f})")

# Save test predictions
test_results_df = df_test.copy()
test_results_df['amd_probability'] = test_probs
test_results_df['predicted_tier'] = tiers
test_results_df['binary_prediction_at_0.5'] = predictions_binary
test_results_df.to_csv(os.path.join(RESULTS_DIR, f'test_predictions_{RUN_ID}.csv'), index=False)
log(f"Test predictions saved")


# ============================================================================
# SECTION 10: DIAGNOSTIC PLOTS
# ============================================================================

log("=== Generating diagnostic plots ===")

# Plot 1: Training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_range, history['val_loss'], label='Val', marker='s')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Loss Curves'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_auroc'], label='Val AUROC', marker='o', color='green')
axes[1].plot(epochs_range, history['val_auprc'], label='Val AUPRC', marker='s', color='orange')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(epochs_range, history['lr'], marker='.')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate')
axes[2].set_title('LR Schedule'); axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'training_curves_{RUN_ID}.png'), dpi=150)
plt.close()

# Plot 2: ROC and Precision-Recall curves on test set
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# ROC
axes[0].plot(fpr, tpr, label=f'AUROC = {test_auroc:.3f}', linewidth=2)
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.3, label='Random')
axes[0].scatter(1 - specificity, sensitivity, s=100, c='red', zorder=5,
                label=f'Threshold=0.5 (Sens={sensitivity:.2f}, Spec={specificity:.2f})')
axes[0].scatter(fpr[idx_95spec], tpr[idx_95spec], s=100, c='green', zorder=5,
                label=f'95% Spec point (Sens={sens_at_95spec:.2f})')
axes[0].set_xlabel('False Positive Rate (1 - Specificity)')
axes[0].set_ylabel('True Positive Rate (Sensitivity)')
axes[0].set_title('Test Set ROC Curve')
axes[0].legend(loc='lower right'); axes[0].grid(True, alpha=0.3)

# Precision-Recall
precision, recall, _ = precision_recall_curve(test_labels, test_probs)
axes[1].plot(recall, precision, label=f'AUPRC = {test_auprc:.3f}', linewidth=2, color='orange')
baseline = test_labels.sum() / len(test_labels)
axes[1].axhline(baseline, color='k', linestyle='--', alpha=0.3, label=f'Baseline (prevalence={baseline:.2f})')
axes[1].set_xlabel('Recall (Sensitivity)')
axes[1].set_ylabel('Precision (PPV)')
axes[1].set_title('Test Set Precision-Recall Curve')
axes[1].legend(loc='upper right'); axes[1].grid(True, alpha=0.3)

# Probability distributions by class
axes[2].hist(test_probs[test_labels == 0], bins=40, alpha=0.5, label='Non-AMD', color='blue', density=True)
axes[2].hist(test_probs[test_labels == 1], bins=40, alpha=0.5, label='AMD', color='red', density=True)
axes[2].axvline(low_t, color='orange', linestyle='--', label=f'Uncertain lower ({low_t})')
axes[2].axvline(high_t, color='red', linestyle='--', label=f'Uncertain upper ({high_t})')
axes[2].set_xlabel('Predicted AMD Probability')
axes[2].set_ylabel('Density')
axes[2].set_title('Prediction Distribution by True Class')
axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, f'test_analysis_{RUN_ID}.png'), dpi=150)
plt.close()

log("Diagnostic plots saved")


# ============================================================================
# SECTION 11: GRAD-CAM SANITY CHECK
# ============================================================================

log("=== Running Grad-CAM sanity check ===")
log("For AMD, attention should land on the MACULA (center of image)")

try:
    from pytorch_grad_cam import GradCAM
    from pytorch_grad_cam.utils.image import show_cam_on_image

    target_layer = model.backbone.conv_head if hasattr(model.backbone, 'conv_head') else model.backbone.blocks[-1]
    cam = GradCAM(model=model, target_layers=[target_layer])

    # Sample: highest-probability AMD, mid-probability, and lowest (confident non-AMD)
    sorted_probs = np.argsort(test_probs)
    high_conf_amd_idx = sorted_probs[-3:]         # top 3 predicted AMD
    high_conf_normal_idx = sorted_probs[:3]        # top 3 predicted normal
    mid_prob_idx = sorted_probs[len(sorted_probs)//2 - 1:len(sorted_probs)//2 + 2]  # middle 3

    all_display_indices = list(high_conf_amd_idx) + list(mid_prob_idx) + list(high_conf_normal_idx)
    n_samples = len(all_display_indices)
    categories = ['High-prob AMD']*3 + ['Uncertain']*3 + ['High-prob Normal']*3

    fig, axes = plt.subplots(n_samples, 2, figsize=(10, 4 * n_samples))
    for i, (idx, cat) in enumerate(zip(all_display_indices, categories)):
        image, actual_label = test_dataset[idx]
        input_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():
            pred_logit = model(input_tensor).item()
            pred_prob = 1 / (1 + np.exp(-pred_logit))

        grayscale_cam = cam(input_tensor=input_tensor)[0]
        display = image.permute(1, 2, 0).cpu().numpy()
        display = (display * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1)
        visualization = show_cam_on_image(display, grayscale_cam, use_rgb=True)

        actual_str = "AMD" if actual_label.item() == 1 else "Normal"
        axes[i, 0].imshow(display)
        axes[i, 0].set_title(f"{cat}\nActual: {actual_str} | Predicted prob: {pred_prob:.3f}")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(visualization)
        axes[i, 1].set_title("Attention (should focus on macula for AMD)")
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'gradcam_sanity_{RUN_ID}.png'), dpi=150)
    plt.close()
    log("Grad-CAM plots saved. Inspect: AMD-predicted images should show attention on macula (image center)")

except Exception as e:
    log(f"Grad-CAM failed (non-fatal): {e}")


# ============================================================================
# SECTION 12: FINAL SUMMARY
# ============================================================================

summary = {
    'run_id': RUN_ID,
    'config': CONFIG,
    'dataset': 'ODIR-5K',
    'task': 'AMD binary classification',
    'best_val_auroc': float(best_val_auroc),
    'test_auroc': float(test_auroc),
    'test_auprc': float(test_auprc),
    'test_sensitivity_at_0.5': float(sensitivity),
    'test_specificity_at_0.5': float(specificity),
    'test_ppv_at_0.5': float(ppv),
    'test_npv_at_0.5': float(npv),
    'test_sensitivity_at_95_specificity': float(sens_at_95spec),
    'test_threshold_at_95_specificity': float(thr_at_95spec),
    'n_train': len(df_train),
    'n_val': len(df_val),
    'n_test': len(df_test),
    'amd_prevalence_train': float(df_train['amd'].mean()),
    'amd_prevalence_test': float(df_test['amd'].mean()),
    'n_epochs_completed': len(history['train_loss']),
    'total_training_minutes': total_time / 60,
    'tier_percentages': {tier: float(tier_counts.get(tier, 0) / len(tiers) * 100)
                         for tier in ['Normal', 'Uncertain', 'AMD']},
    'files': {
        'best_model': os.path.join(CHECKPOINT_DIR, 'best_model.pth'),
        'test_predictions': os.path.join(RESULTS_DIR, f'test_predictions_{RUN_ID}.csv'),
        'training_curves': os.path.join(RESULTS_DIR, f'training_curves_{RUN_ID}.png'),
        'test_analysis': os.path.join(RESULTS_DIR, f'test_analysis_{RUN_ID}.png'),
        'gradcam': os.path.join(RESULTS_DIR, f'gradcam_sanity_{RUN_ID}.png'),
        'log': RUN_LOG,
    }
}

summary_path = os.path.join(RESULTS_DIR, f'summary_{RUN_ID}.json')
with open(summary_path, 'w') as f:
    json.dump(summary, f, indent=2)

log(f"=== RUN COMPLETE ===")
log(f"Summary: {summary_path}")
log(f"Best model: {os.path.join(CHECKPOINT_DIR, 'best_model.pth')}")
log(f"")
log(f"Next steps:")
log(f"  1. Inspect Grad-CAM: AMD-predicted images should highlight macula")
log(f"  2. Check ROC curve: AUROC > 0.85 is good for ODIR-5K scale, > 0.90 is strong")
log(f"  3. Review confusion matrix — sensitivity matters most for screening")
log(f"  4. If AUROC < 0.80, consider: more epochs, combining with iChallenge-AMD data,")
log(f"     or checking that class weighting worked (batches should look ~50/50 AMD)")
log(f"  5. Convert best_model.pth to ONNX and integrate into FundusView multi-head pipeline")

[20:33:27] === AMD training run 20260908_203327 started ===
[20:33:27] Config: {
  "backbone": "efficientnet_b2",
  "image_size": 512,
  "batch_size": 16,
  "num_epochs": 40,
  "learning_rate": 0.0001,
  "weight_decay": 0.0001,
  "num_workers": 2,
  "seed": 42,
  "val_split": 0.15,
  "test_split": 0.15,
  "use_pretrained": true,
  "dropout": 0.3,
  "early_stopping_patience": 10,
  "uncertain_lower_threshold": 0.3,
  "uncertain_upper_threshold": 0.6,
  "oversample_amd": true
}
[20:33:30] GPU: Tesla T4, VRAM: 15.6 GB
[20:33:30] Using image directory: /content/drive/MyDrive/FundusView/ODIR-5K/preprocessed_images
[20:33:30] Using label file: /content/drive/MyDrive/FundusView/ODIR-5K/full_df.csv
[20:33:30] Loaded labels: 6392 rows
[20:33:30] Columns: ['ID', 'Patient Age', 'Patient Sex', 'Left-Fundus', 'Right-Fundus', 'Left-Diagnostic Keywords', 'Right-Diagnostic Keywords', 'N', 'D', 'G', 'C', 'A', 'H', 'M', 'O', 'filepath', 'labels', 'target', 'filename']
[20:33:30] Detected patient-level f

[20:41:28] Epoch 1/40 | Train loss: 0.2661 | Val loss: 0.3314 | Val AUROC: 0.8449 | Val AUPRC: 0.4582 | LR: 9.98e-05 | Time: 477s
[20:41:29]   → New best val AUROC 0.8449 saved to best_model.pth


[20:47:46] Epoch 2/40 | Train loss: 0.0790 | Val loss: 0.2544 | Val AUROC: 0.8803 | Val AUPRC: 0.5259 | LR: 9.94e-05 | Time: 377s
[20:47:47]   → New best val AUROC 0.8803 saved to best_model.pth


[20:54:05] Epoch 3/40 | Train loss: 0.0527 | Val loss: 0.2465 | Val AUROC: 0.8536 | Val AUPRC: 0.5939 | LR: 9.86e-05 | Time: 378s


[21:00:23] Epoch 4/40 | Train loss: 0.0345 | Val loss: 0.2585 | Val AUROC: 0.8513 | Val AUPRC: 0.5535 | LR: 9.76e-05 | Time: 378s


...

[21:57:09] Epoch 13/40 | Train loss: 0.0065 | Val loss: 0.2775 | Val AUROC: 0.8664 | Val AUPRC: 0.6204 | LR: 7.61e-05 | Time: 380s


[22:03:27] Epoch 14/40 | Train loss: 0.0107 | Val loss: 0.3441 | Val AUROC: 0.8564 | Val AUPRC: 0.5461 | LR: 7.27e-05 | Time: 378s


[22:09:47] Epoch 15/40 | Train loss: 0.0045 | Val loss: 0.3346 | Val AUROC: 0.8603 | Val AUPRC: 0.5799 | LR: 6.91e-05 | Time: 380s


[22:16:04] Epoch 16/40 | Train loss: 0.0083 | Val loss: 0.4179 | Val AUROC: 0.8649 | Val AUPRC: 0.5567 | LR: 6.55e-05 | Time: 377s
[22:16:05] Early stopping: no AUROC improvement for 10 epochs
[22:16:05] === Training complete in 102.6 minutes ===
[22:16:05] Best validation AUROC: 0.8878
[22:16:05] === Loading best model for test evaluation ===


UnpicklingError: Weights only load failed. This file can still be loaded, to do so you have two options, [1mdo those steps only if you trust the source of the checkpoint[0m. 
	(1) In PyTorch 2.6, we changed the default value of the `weights_only` argument in `torch.load` from `False` to `True`. Re-running `torch.load` with `weights_only` set to `False` will likely succeed, but it can result in arbitrary code execution. Do it only if you got the file from a trusted source.
	(2) Alternatively, to load with `weights_only=True` please check the recommended steps in the following error message.
	WeightsUnpickler error: Unsupported global: GLOBAL numpy._core.multiarray.scalar was not an allowed global by default. Please use `torch.serialization.add_safe_globals([numpy._core.multiarray.scalar])` or the `torch.serialization.safe_globals([numpy._core.multiarray.scalar])` context manager to allowlist this global if you trust this class/function.

Check the documentation of torch.load to learn 

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
